# Original Datasets

We generate train/test splits by selecting every 8th image as test data. The split metadata is saved as JSON, and the final splits are stored in a GS-W-compatible `.tsv` file.

In [ ]:
import json
from pathlib import Path

# Update this path to the directory containing the scene folders.
# Expected structure: DATASETS_ROOT / "figurines", "ramen", "teatime", ...
DATASETS_ROOT = Path("../datasets").expanduser().resolve()


def find_image_folder(scene_dir):
    # Case 1: standard structure
    if (scene_dir / "images").exists():
        return scene_dir / "images"

    # Case 2: images directly in root
    image_files = list(scene_dir.glob("*.jpg")) + list(scene_dir.glob("*.png"))
    if len(image_files) > 0:
        return scene_dir

    raise AssertionError(f"No images found in {scene_dir}")


def create_gsw_tsv_split(
    scene,
    base_dir=DATASETS_ROOT,
    test_stride=8,
    image_exts=(".jpg", ".jpeg", ".png"),
):
    scene_dir = Path(base_dir) / scene
    assert scene_dir.exists(), f"{scene_dir} not found"

    image_dir = find_image_folder(scene_dir)
    print(f"[{scene}] Using image folder: {image_dir}")

    image_paths = sorted([
        p for p in image_dir.iterdir()
        if p.suffix.lower() in image_exts
    ], key=lambda p: p.stem)

    rows = []
    split_json = {"train": [], "test": []}

    for idx, img_path in enumerate(image_paths):
        split = "test" if idx % test_stride == 0 else "train"

        rows.append({
            "filename": img_path.name,
            "split": split,
        })

        split_json[split].append(img_path.name)

    # Save TSV (GS-W expects it here)
    tsv_path = scene_dir / f"{scene}.tsv"
    with open(tsv_path, "w") as f:
        f.write("filename\tsplit\n")
        for row in rows:
            f.write(f"{row['filename']}\t{row['split']}\n")

    # Save JSON (for your scripts)
    json_path = scene_dir / "split.json"
    with open(json_path, "w") as f:
        json.dump(split_json, f, indent=2)

    print(f"Saved TSV:  {tsv_path}")
    print(f"Saved JSON: {json_path}")
    print(f"{scene}: {len(split_json['train'])} train, {len(split_json['test'])} test")

    return split_json

In [ ]:
BASE = DATASETS_ROOT

create_gsw_tsv_split("figurines", BASE)
create_gsw_tsv_split("ramen", BASE)
create_gsw_tsv_split("teatime", BASE)

# Appearance-varied Datasets

Here, the original datasets are synthetically made more “in-the-wild” by applying appearance variations to a random 40% subset of the training images, using `seed 42`.

The applied transformations include brightness, contrast, color temperature, gamma, blur, noise, and mixed lighting changes. Test images are kept unchanged to preserve a clean evaluation split.

For reproducibility, a `variation_log.json` file is also saved. This file records which images were modified, which transformation was applied, and the corresponding parameter values.

In [ ]:
import shutil
from pathlib import Path

BASE = DATASETS_ROOT

def copy_for_varied(scene, suffix="_varied", overwrite=False):
    src = BASE / scene
    dst = BASE / f"{scene}{suffix}"

    assert src.exists(), f"Source does not exist: {src}"

    if dst.exists():
        if overwrite:
            print(f"Removing existing: {dst}")
            shutil.rmtree(dst)
        else:
            raise FileExistsError(f"{dst} already exists. Use overwrite=True if intentional.")

    print(f"Copying {src} -> {dst}")
    shutil.copytree(src, dst)

    # Rename TSV if needed: figurines.tsv -> figurines_varied.tsv is NOT necessary.
    # Your GS-W code just searches for any *.tsv in the dataset folder.
    print(f"Done: {dst}")

In [ ]:
copy_for_varied("figurines", overwrite=True)
copy_for_varied("ramen", overwrite=True)
copy_for_varied("teatime", overwrite=True)

In [ ]:
import json
import random
from pathlib import Path
from PIL import Image, ImageEnhance, ImageFilter
import numpy as np

BASE = DATASETS_ROOT

def find_image_folder(scene_dir):
    if (scene_dir / "images").exists():
        return scene_dir / "images"

    image_files = list(scene_dir.glob("*.jpg")) + list(scene_dir.glob("*.jpeg")) + list(scene_dir.glob("*.png"))
    if image_files:
        return scene_dir

    raise FileNotFoundError(f"No image folder or root images found in {scene_dir}")


def apply_appearance_variation(img, seed):
    random.seed(seed)
    np.random.seed(seed)

    img = img.convert("RGB")
    arr = np.asarray(img).astype(np.float32) / 255.0

    recipe = random.choice([
        "bright_warm",
        "dark_cool",
        "low_contrast",
        "high_contrast",
        "gamma_shift",
        "color_shift",
        "mild_blur",
        "noise",
        "mixed_light",
    ])

    params = {"recipe": recipe}

    if recipe == "bright_warm":
        brightness = random.uniform(1.15, 1.35)
        contrast = random.uniform(1.00, 1.15)
        arr[..., 0] *= random.uniform(1.05, 1.18)
        arr[..., 2] *= random.uniform(0.85, 0.98)
        img = Image.fromarray((np.clip(arr, 0, 1) * 255).astype(np.uint8))
        img = ImageEnhance.Brightness(img).enhance(brightness)
        img = ImageEnhance.Contrast(img).enhance(contrast)
        params.update(brightness=brightness, contrast=contrast)

    elif recipe == "dark_cool":
        brightness = random.uniform(0.65, 0.85)
        contrast = random.uniform(0.90, 1.10)
        arr[..., 0] *= random.uniform(0.85, 0.98)
        arr[..., 2] *= random.uniform(1.05, 1.20)
        img = Image.fromarray((np.clip(arr, 0, 1) * 255).astype(np.uint8))
        img = ImageEnhance.Brightness(img).enhance(brightness)
        img = ImageEnhance.Contrast(img).enhance(contrast)
        params.update(brightness=brightness, contrast=contrast)

    elif recipe == "low_contrast":
        contrast = random.uniform(0.65, 0.85)
        brightness = random.uniform(0.90, 1.10)
        img = ImageEnhance.Contrast(img).enhance(contrast)
        img = ImageEnhance.Brightness(img).enhance(brightness)
        params.update(contrast=contrast, brightness=brightness)

    elif recipe == "high_contrast":
        contrast = random.uniform(1.20, 1.45)
        saturation = random.uniform(0.90, 1.20)
        img = ImageEnhance.Contrast(img).enhance(contrast)
        img = ImageEnhance.Color(img).enhance(saturation)
        params.update(contrast=contrast, saturation=saturation)

    elif recipe == "gamma_shift":
        gamma = random.uniform(0.65, 1.45)
        arr = np.power(arr, gamma)
        img = Image.fromarray((np.clip(arr, 0, 1) * 255).astype(np.uint8))
        params.update(gamma=gamma)

    elif recipe == "color_shift":
        scale = np.random.uniform(0.80, 1.25, size=3)
        bias = np.random.uniform(-0.08, 0.08, size=3)
        arr = arr * scale + bias
        img = Image.fromarray((np.clip(arr, 0, 1) * 255).astype(np.uint8))
        params.update(rgb_scale=scale.tolist(), rgb_bias=bias.tolist())

    elif recipe == "mild_blur":
        radius = random.uniform(0.4, 1.2)
        brightness = random.uniform(0.90, 1.15)
        img = img.filter(ImageFilter.GaussianBlur(radius=radius))
        img = ImageEnhance.Brightness(img).enhance(brightness)
        params.update(radius=radius, brightness=brightness)

    elif recipe == "noise":
        sigma = random.uniform(0.015, 0.045)
        noise = np.random.normal(0, sigma, arr.shape)
        arr = arr + noise
        img = Image.fromarray((np.clip(arr, 0, 1) * 255).astype(np.uint8))
        params.update(noise_sigma=sigma)

    elif recipe == "mixed_light":
        brightness = random.uniform(0.75, 1.30)
        contrast = random.uniform(0.80, 1.35)
        saturation = random.uniform(0.75, 1.30)
        gamma = random.uniform(0.80, 1.25)

        arr = np.power(arr, gamma)
        img = Image.fromarray((np.clip(arr, 0, 1) * 255).astype(np.uint8))
        img = ImageEnhance.Brightness(img).enhance(brightness)
        img = ImageEnhance.Contrast(img).enhance(contrast)
        img = ImageEnhance.Color(img).enhance(saturation)

        params.update(
            brightness=brightness,
            contrast=contrast,
            saturation=saturation,
            gamma=gamma,
        )

    return img, params


def apply_variations_to_existing_varied_dataset(
    scene_varied,
    base_dir=BASE,
    replace_fraction=0.40,
    seed=42,
    image_exts=(".jpg", ".jpeg", ".png"),
):
    scene_dir = Path(base_dir) / scene_varied
    assert scene_dir.exists(), f"Dataset not found: {scene_dir}"

    split_path = scene_dir / "split.json"
    assert split_path.exists(), f"split.json not found in {scene_dir}"

    with open(split_path, "r") as f:
        split = json.load(f)

    train_names = set(split["train"])
    test_names = set(split["test"])

    image_dir = find_image_folder(scene_dir)

    image_paths = sorted([
        p for p in image_dir.iterdir()
        if p.suffix.lower() in image_exts
    ], key=lambda p: p.stem)

    train_image_paths = [p for p in image_paths if p.name in train_names]
    test_image_paths = [p for p in image_paths if p.name in test_names]

    assert len(train_image_paths) > 0, "No train images found. Check split.json filenames."
    assert len(test_image_paths) > 0, "No test images found. Check split.json filenames."

    random.seed(seed)

    n_replace = int(len(train_image_paths) * replace_fraction)
    selected = random.sample(train_image_paths, n_replace)

    log = {
        "dataset": scene_varied,
        "replace_fraction": replace_fraction,
        "seed": seed,
        "num_total_images": len(image_paths),
        "num_train_images": len(train_image_paths),
        "num_test_images_clean_untouched": len(test_image_paths),
        "num_train_images_modified": n_replace,
        "modified_images": {},
        "untouched_test_images": sorted([p.name for p in test_image_paths]),
    }

    for idx, img_path in enumerate(selected):
        img = Image.open(img_path)
        varied_img, params = apply_appearance_variation(img, seed + idx)
        varied_img.save(img_path)
        log["modified_images"][img_path.name] = params

    log_path = scene_dir / "variation_log.json"
    with open(log_path, "w") as f:
        json.dump(log, f, indent=2)

    print(f"Done: {scene_varied}")
    print(f"Modified train images: {n_replace}/{len(train_image_paths)}")
    print(f"Test images left clean: {len(test_image_paths)}")
    print(f"Saved log: {log_path}")

    return log

In [ ]:
apply_variations_to_existing_varied_dataset("figurines_varied", replace_fraction=0.40, seed=42)
apply_variations_to_existing_varied_dataset("ramen_varied", replace_fraction=0.40, seed=42)
apply_variations_to_existing_varied_dataset("teatime_varied", replace_fraction=0.40, seed=42)

# Optional COLMAP Check

This section checks whether `test_*` images appear in the COLMAP reconstruction metadata. Update `GSW_ROOT` if your GS-W checkout is stored elsewhere.

In [ ]:
import sys
from pathlib import Path

# Optional: point this to a local GS-W checkout if you want to run the COLMAP check below.
GSW_ROOT = Path("../Stage-1/Gaussian-in-the-Wild").expanduser().resolve()
sys.path.append(str(GSW_ROOT))

In [ ]:
import sys
sys.path.append(str(GSW_ROOT / "scene"))

from colmap_loader import read_extrinsics_binary
from pathlib import Path

def check_test_images_in_colmap(scene, base=DATASETS_ROOT):
    sparse_path = Path(base) / scene / "sparse/0/images.bin"

    extrinsics = read_extrinsics_binary(sparse_path)

    image_names = [extr.name for extr in extrinsics.values()]
    test_images = [name for name in image_names if name.startswith("test_")]
    frame_images = [name for name in image_names if name.startswith("frame_")]

    print(f"\nScene: {scene}")
    print(f"Total COLMAP images: {len(image_names)}")
    print(f"frame_* images: {len(frame_images)}")
    print(f"test_* images: {test_images}")

In [ ]:
check_test_images_in_colmap("figurines")
check_test_images_in_colmap("ramen")
check_test_images_in_colmap("teatime")